# 89장 · 자료 접근 범위와 대체 데이터

자료마다 접근할 수 있는 사용자가 다릅니다. 자료 목록에 허용 범위를 적고 외부 연결이 끊겼을 때 사용할 근거를 정합니다.

서버가 읽을 수 있는 규정과 공개 사용자가 인용할 수 있는 규정의 범위를 분리합니다. 공개 그룹으로 필터링한 자료에서 근거 위치를 복원합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 인수 계약을 공개 자료와 대조하기

88장 인수 조건에 적은 필드가 공개 시연용 자료에 있는지 검사합니다. 자료를 추가했다면 정답으로 가리킨 ID가 공개 범위에서 실제로 열리는지도 봅니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter88.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter89-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter88.json"
sample_file = here / "chapter89-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['contracts'])

교재 예시 chapter89-start.json
[{'id': 'Q01', 'q': '장비 A의 점검 전 확인 사항은?', 'status': 'answered', 'source_id': 'RULE-A', 'expected_fields': ['id', 'source', 'page', 'section', 'revised_on', 'version']}, {'id': 'Q02', 'q': '점검 전 확인 사항은?', 'status': 'need_clarification', 'source_id': None, 'expected_fields': []}, {'id': 'Q03', 'q': '장비 A를 다음 달에 교체하나요?', 'status': 'no_evidence', 'source_id': None, 'expected_fields': []}]


두 번째 줄은 읽은 파일의 `contracts` 값입니다.

## 공개 범위별 문서 목록

자료 카드 A　RULE-A / 가상 장비실 규정 A / 1쪽 2.1절 / 개정 2026-11-01 / rules-v1 / public·admin 허용. 자료 카드 B　RULE-B / 가상 장비실 규정 B / 2쪽 2.2절 / 같은 개정일·판본 / 같은 두 그룹 허용. 별도 내부 규정 RULE-ADMIN은 admin만 읽도록 정해 둡니다. 세 문서 모두 수업용 가상 자료이며 소속 기관의 실제 방침은 별도로 확인해야 합니다.


### 셀 01 · 공개 그룹의 자료 필터

자료 담당자가 가상 public 사용자에게 제공할 문서 ID만 고릅니다.


In [1]:
documents = [{'id': 'RULE-A',
  'equipment': '장비 A',
  'text': '점검 전 전원 표시등을 확인합니다.',
  'source': '가상 장비실 규정 A',
  'page': 1,
  'section': '2.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-B',
  'equipment': '장비 B',
  'text': '점검 전 고정 장치를 확인합니다.',
  'source': '가상 장비실 규정 B',
  'page': 2,
  'section': '2.2',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-ADMIN',
  'equipment': None,
  'text': '관리자 전용 절차는 선임 검토 기록을 확인하는 것입니다.',
  'source': '가상 장비실 내부 규정',
  'page': 3,
  'section': '3.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['admin']}]
allowed = [doc for doc in documents if "public" in doc["groups"]]
print([doc["id"] for doc in allowed])

['RULE-A', 'RULE-B']


RULE-A와 RULE-B 두 개가 남습니다. 검색한 뒤 민감한 결과를 가리는 방식에서는 검색 기록에 본문이 남을 수 있으므로 이 예시에서는 먼저 후보를 줄입니다.

검색 전에 공개 자료를 걸러 낸 결과가 RULE-A와 RULE-B입니다. 이 목록은 접근할 수 있는 후보를 정한 것이고, 어느 문서가 이번 질문의 근거인지는 다음 판단입니다. 권한 필터와 검색 결과를 따로 기록해야 답이 비었을 때 어느 쪽에서 제외됐는지 추적할 수 있습니다.


### 허용 문서 목록에서 빠진 항목 추적하기

출력된 RULE-A와 RULE-B는 public 사용자가 살펴볼 수 있는 후보입니다. 여기에서 RULE-ADMIN이 빠진 이유를 찾으려면 원자료의 groups를 읽어 보세요. admin만 들어 있으므로 장비 이름을 비교하기 전에 이미 제외됩니다.


## 연동 실패 시 사용할 대체 자료

접근 기록 예시　대상 가상 장비실 규정 A / 자료 형태 교육용 JSON / 읽은 위치 1쪽 2.1절 / 입력 장비 A / 관찰 “점검 전 전원 표시등을 확인합니다.” / 실제 장비실 시스템 접근 미확인. 원격 연결을 확인하지 못한 팀도 같은 필드의 가상 레코드로 질문 흐름을 설계할 수 있습니다. 연결 상태와 데이터 읽기 상태는 따로 남기세요.


### 팀 참고 예시 · 근거 문서의 위치


In [2]:
selected = "장비 B"
locator = next(doc for doc in allowed if doc["equipment"] == selected)
print(locator["source"], locator["page"], locator["section"])

가상 장비실 규정 B 2 2.2


가상 장비실 규정 B의 2쪽 2.2절을 확인합니다. 자료 ID만 발견한 경우와 원문 위치까지 읽은 경우를 목록에서 구분할 수 있습니다. 인수표에는 ‘읽은 공개 파일과 문서 ID’, ‘실제 시스템 접근 승인 여부’를 두 칸으로 나눠 적으세요. 파일만 열었다면 두 번째 칸은 미확인입니다.


#### 대체 자료로 시연할 때

<mark>대체 문서의 접근 범위와 원문 위치도 기록해야 합니다.</mark> 연동이 막혀 RULE-B 예시를 쓴다면 학생이 어디까지 실제로 조회했는지 분명히 하세요. 대체 문서에 없는 필드까지 원래 서비스에서 확인한 것으로 쓰면 안 됩니다.

선수 개념 · 88장 인수 기준과 작업 분해

## 공개 시연용 파일에 넣을 열과 행 고르기

88장에서 확인한 원본에는 화면에 필요한 출처 외에 접근 그룹도 있습니다. 팀 밖의 동료에게 조회 샘플을 줄 때 공개할 레코드와 전달할 열을 차례로 고릅니다.

### 셀 04 · 공개 규정 두 건으로 인계 파일 만들기

노트북의 세 규정을 JSON Lines 형식으로 바꿉니다. 한 줄은 문서 한 건입니다. 만든 문자열을 다시 열어 파일을 받는 사람이 볼 수 있는 문서와 쪽을 확인하세요.

In [3]:
import json

public_fields = ("id", "text", "source", "page", "section", "revised_on", "version")
public_lines = [
    json.dumps({key: row[key] for key in public_fields}, ensure_ascii=False)
    for row in documents if "public" in row["groups"]
]
package = "\n".join(public_lines)
reopened = [json.loads(line) for line in package.splitlines()]
print([(row["id"], row["page"]) for row in reopened])
assert all(set(row) == set(public_fields) for row in reopened)

[('RULE-A', 1), ('RULE-B', 2)]


받는 쪽에는 RULE-A의 1쪽, RULE-B의 2쪽만 있습니다. 행을 추릴 때 groups를 썼지만 전달한 열에는 groups가 없습니다.

### 연동이 다시 가능해질 때 대조할 것

실제 원문 접근이 열리면 ID별로 쪽, 절, 개정일을 대조합니다. 대체 샘플을 먼저 받은 동료는 파일 앞의 두 ID와 자료판 rules-v1을 기록해 두세요. 다른 판본이면 평가 결과를 한 표에 합치기 전에 바뀐 문서부터 확인합니다.

<mark>어느 행을 읽을 수 있는지와 그 행의 어느 열을 전달할지는 별도로 정합니다.</mark> 공개 규정 두 건을 골라도 내부 메모 열까지 그대로 복사하면 시연에 필요 없는 정보가 함께 나갑니다.

90장에서는 이 두 공개 문서의 ID를 기대 출처로 쓰고, 자료 밖 질문에는 기대 출처를 비워 둡니다.

## 팀 사례에 적용하기 · 인수 계약을 공개 자료와 대조하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
equipment_names = {row["id"]: row["equipment"] for row in allowed}
public_docs = {doc["id"]: {**doc, "equipment": equipment_names[doc["id"]]} for doc in reopened}
missing = [row["id"] for row in incoming["contracts"] if row["source_id"] and
           not all(f in public_docs.get(row["source_id"], {}) for f in row["expected_fields"])]
new = {"contracts": incoming["contracts"], "documents": public_docs, "missing": missing}
print("계약을 충족하지 못한 문항", missing)

계약을 충족하지 못한 문항 []


누락 목록에 ID가 보이면 시연 전에 팀 문서를 바꾸거나 해당 질문을 보류합니다. 자료 목록이 열렸다는 사실만으로 외부 장비 시스템의 접근을 확인할 수는 없습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter89.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter89.json


90장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

자료 목록에 문서 ID·판본·허용 그룹을 남기고, 화면에 보여 줄 수 있는 문장 하나를 붙이세요. 90장은 이 목록에서 기대 출처를 고릅니다.
